In [1]:
from functions import ping_openai, fetch_articles, read_prompt_from_file_only, load_function_schema, normalize_id, normalize_type, combine_paragraphs, format_nodes_for_prompt, get_schema
from model_dictionary import model_dictionary
from inputs import relationship_groups, groups_to_prompts, nodes_by_group_prompt

import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from openai import OpenAI
from dotenv import load_dotenv
import json
from bson import json_util
import re

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)
    
ping_openai(client)

# DO WE NEED THIS FUNCTION STILL?
#articles = fetch_articles(articles_collection, limit=1,offset=15)

✅ Connected to MongoDB Atlas! Database: tuone
✅ Successfully connected to OpenAI API!
Available Models: ['o1-pro', 'gpt-4o-realtime-preview-2024-12-17', 'o1-pro-2025-03-19', 'gpt-4o-audio-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4o-transcribe', 'gpt-4o-mini-transcribe', 'gpt-4o-realtime-preview', 'babbage-002', 'gpt-4o-mini-tts', 'tts-1-hd-1106', 'text-embedding-3-large', 'gpt-4', 'text-embedding-ada-002', 'computer-use-preview', 'computer-use-preview-2025-03-11', 'tts-1-hd', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4o-mini-realtime-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'tts-1-1106', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4-turbo', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'chatgpt-4o-latest', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4o-2024-11-20', 'whisper-1', 'gpt-4o-2024-05-13

In [2]:
def extract_nodes(article_id, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_name):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            functions = [function_schema],
            function_call = {"name": "extract_clean_tech_entities"},
            temperature=0
        )

        # parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments

        #an_example_output_for_finetuning = {
        #    "messages": [
        #        {"role": "system", "content": prompt},
        #        {"role": "user", "content": f"Here is the article: {text}"},
        #        {"role": "assistant", "content": function_args}
        #    ]
        #}

        #with open("example_for_finetuning.json", "w", encoding="utf-8") as f:
        #    json.dump(example_for_finetuning, f, ensure_ascii=False, indent=2)

        extracted_data = json.loads(function_args)
        print(f"✅ Succesful call to OpenAI. The following nodes are returned: {extracted_data}\n")

        if "nodes" not in extracted_data:
            raise ValueError("❌ Expected 'nodes' key in LLM output but not found.")

        nodes_by_category = extracted_data["nodes"]

        # Flatten nodes while retaining their type
        formatted_nodes = []
        for node_type, node_categories in nodes_by_category.items():
            for node in node_categories:
                formatted_node = {
                    "article_id": article_id,
                    "id": normalize_id(node.get("id")),
                    "type": normalize_type(node.get("type")),
                    "name": node.get("name"),
                    "location": {
                        "city": node.get("location", {}).get("city", ""),
                        "country": node.get("location", {}).get("country", "")
                    } if node.get("location") else None,
                    "amount": node.get("amount"), #possibly to drop
                    "status": node.get("status"),
                }
                formatted_nodes.append(formatted_node)

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []

In [4]:
def extract_relationships(article_id, text, nodes, relationship_group, model_name, allowed_types=None):

    PROMPT_PATH = groups_to_prompts[relationship_group]
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    
    allowed_types = nodes_by_group_prompt[relationship_group]
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    
    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        #parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Succesful openAI call: returned the following relationships {extracted_data}\n")

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []
        for rel in extracted_data["relationships"]:
            raw_source = rel.get("source")
            raw_target = rel.get("target")
            norm_source = normalize_id(raw_source)
            norm_target = normalize_id(raw_target)

            #print(f"🔍 Normalizing IDs - Source: {raw_source} → {norm_source}, Target: {raw_target} → {norm_target}")

            formatted_relationships.append({
                "article_id": article_id,
                "id": f"{norm_source}_{rel.get('type')}_{norm_target}",
                "source": norm_source,
                "target": norm_target, # should this be changed to raw_target?
                "type": rel.get("type"),
                "group": relationship_group
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []

In [5]:
def process_articles(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find(
            {"meta.ID": {"$regex": "^27"}},  
            {"_id": 1, "paragraphs": 1, "validation": 1}       
        )
        .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)    # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  #convert ObjectId to string

        if article.get("validation") is True:
            print(f"⏭️  Skipping Article ID: {articleID} - Article is validated")
            continue

        text = combine_paragraphs(article)
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary["nodes"])

            all_relationships = []
            for relationship_group,model_name in model_dictionary.items():
                if relationship_group == "nodes":
                    continue  # ✅ skip node model, not a relationship group

                print("- - - Querying openai for relationships: , using model:  - - - ", relationship_group, model_name)
                allowed_types = relationship_groups[relationship_group]
                print("Node types included in the query: ", allowed_types)
                relationships = extract_relationships(articleID, text, formatted_nodes, relationship_group, model_name, allowed_types)
                all_relationships.extend(relationships)

            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  #match by article id
                {"$set": {
                    "nodes": formatted_nodes or [],
                    "relationships": all_relationships or []
                    }})

            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")

In [6]:
n_articles = 1
offset_articles = 0

# named entity recognition (like company, factory, investment)
PROMPT_PATH = "prompts/entities-only.txt"
FUNCTION_SCHEMA_PATH = "schemas/entities.json"

process_articles(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary)

🔹 Processing 1 articles (Offset: 0)
📌 Processing Article ID: 67f52d07981040986eab7b6c
✅ Succesful call to OpenAI. The following nodes are returned: {'nodes': {'companies': [{'id': 'company_volkswagen', 'type': 'company', 'name': 'Volkswagen'}], 'factories': [{'id': 'factory_zwickau', 'type': 'factory', 'name': 'Zwickau', 'location': {'city': 'Zwickau', 'country': 'Germany'}}, {'id': 'factory_wolfsburg', 'type': 'factory', 'name': 'Wolfsburg', 'location': {'city': 'Wolfsburg', 'country': 'Germany'}}, {'id': 'factory_puebla', 'type': 'factory', 'name': 'Puebla', 'location': {'city': 'Puebla', 'country': 'Mexico'}}, {'id': 'factory_emden', 'type': 'factory', 'name': 'Emden', 'location': {'city': 'Emden', 'country': 'Germany'}}, {'id': 'factory_zwickau', 'type': 'factory', 'name': 'Zwickau', 'location': {'city': 'Zwickau', 'country': 'Germany'}}, {'id': 'factory_audi_q4_etron', 'type': 'factory', 'name': 'Audi Q4 e-tron', 'location': {'city': 'null', 'country': 'null'}}, {'id': 'factory_cu

In [7]:
offset = 1
limit = 1

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^27"}},  # <-- Add this filter
        {"_id": 1, "paragraphs": 1, "validation": 1, "meta": 1, "title": 1,
         "nodes":1, "relationships":1}       # Keep projection as is
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset)    # Skip first `offset` articles
    .limit(limit)    # Limit the number of articles
)

articles_to_process

[{'_id': ObjectId('67f52d07981040986eab7b6b'),
  'title': 'Tesla to start production of the Model Y Juniper in January',
  'paragraphs': [{'p1': "Tesla has been producing the Model Y for around five years. So it's time for a facelift, which some market observers had expected as early as 2024. Now the time has supposedly come - Tesla is planning to start mass production of the revised Model Y, codenamed Juniper, at its plant in Shanghai in January.",
    'p2': 'However, there has been no official confirmation from Tesla so far. In addition, reports from Chinese media only refer to the factory in China. However, the Model Y is also built in Grünheide near Berlin, and in the US factories in California and Texas. There is still no indication of a corresponding production changeover at these locations.',
    'p3': 'According to reports from China, Tesla will revise the Model Y in a similar way to theModel 3 Highland facelift. That includes improvements to the interior, exterior, battery cap

In [17]:


def process_relationships(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, relationship_type, model_name, allowed_types=None):
    # Query articles that already have nodes_ben and combined_text

    articles_to_process = list(
        articles_collection.find(
            {
                "nodes_ben": {"$exists": True},
                "combined_text": {"$exists": True},
                "meta.ID": {"$regex": "^27"}  # <-- Only this line is added
            },
            {"_id": -1, "combined_text": 1, "nodes_ben": 1, "events_ben": 1, "validation": 1}
        )
        .sort("_id", -1)
        .skip(offset)
        .limit(limit)
    )

    print(f"🔹 Processing {len(articles_to_process)} articles for relationship extraction (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])
        text = combine_paragraphs(article)

        nodes = article.get("nodes", [])

        if article.get("validation") is True:
            print(f"⏭️  Skipping Article ID: {articleID} - Article is validated")
            continue

        if not text or not nodes:
            print(f"⚠️ Missing text or nodes for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            #extract relationships using previously extracted nodes
            formatted_relationships = extract_relationships(articleID, text, nodes, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types)

            print(f"Debug: Extracted relationships: {formatted_relationships}")

            # Update the article document with extracted relationships
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},
                {"$set": {relationship_type: formatted_relationships or []}} #we use an empty list if no relationships are extracted
            )

            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_relationships)} relationships.")
            else:
                print(f"⚠️ No update performed for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing relationships for Article ID {articleID}: {e}")

In [18]:
def format_events_for_prompt(events):
    lines = ["The following is a list of known events. You MUST ALWAYS use the ID of the event (left value) when referring to it in a relationship:"]
    for event in events:
        event_id = event.get("id")
        event_type = event.get("type")
        event_date = event.get("date")
        if event_id and event_type and event_date:
            lines.append(f"- ID: {event_id}: Event: {event_type} ({event_date})")
    return "\n".join(lines)

def extract_relationships_with_events(article_id, text, nodes, events, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types=None):
    
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)
    
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    print("Allowed types: ", allowed_types)
    print(compact_nodes)
    compact_events = format_events_for_prompt(events)
    print(compact_events)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Here is the list of known events:
                {compact_events}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        # Parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved relationships {extracted_data}\n")

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []

        for rel in extracted_data["relationships"]:
            raw_source = rel.get("source")
            raw_target = rel.get("target")
            norm_source = normalize_id(raw_source)
            norm_target = normalize_id(raw_target)

            print(f"🔍 Normalizing IDs - Source: {raw_source} → {norm_source}, Target: {raw_target} → {norm_target}")

            formatted_relationships.append({
                "article_id": article_id,
                "id": f"{norm_source}_{rel.get('type')}_{norm_target}",
                "source": norm_source,
                "target": norm_target,
                "type": rel.get("type")
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []

In [ ]:


# extract ownership relationships
PROMPT_PATH = "prompts/relations-ownership.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-ownership.json"

# process_relationships(n_articles, offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_ownership", model_name="ft:gpt-4o-mini-2024-07-18:personal:ownership-relations:BIF9Uu4b",
#                       allowed_types = ["company", "joint_venture", "factory"], node_type='entities')

# extract financial relationships (origin)
PROMPT_PATH = "prompts/relations-financial-origin.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-financial-origin.json"

# process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_financial_origin", model_name="gpt-4o",
                    #   allowed_types = ["company", "joint_venture", "investment"], node_type='entities')

# extract financial relationships (technological)
PROMPT_PATH = "prompts/relations-financial-technological.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-financial-technological.json"

# process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_financial_technological", model_name="gpt-4o",
#                       allowed_types = ["investment", "capacity", "product", "factory"], node_type='entities')

# extract technological relationships
PROMPT_PATH = "prompts/relations-technological.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-technological.json"

# process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_technological", model_name="gpt-4o",
#                       allowed_types = ["capacity", "product", "factory"], node_type='entities')




###### EVENTS LOGIC ######

# extract events that happen (new-announcemment, construction-start)
PROMPT_PATH = "prompts/events.txt"
FUNCTION_SCHEMA_PATH = "schemas/events.json"

#process_events(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, 'events_ben', allowed_types=["factory"])

# extract chronology of investment events
PROMPT_PATH = "prompts/chronology-investment.txt"
FUNCTION_SCHEMA_PATH = "schemas/chronology-investment.json"

#process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_chronology_investment", allowed_types=["investment"], node_type = "events")

# extract chronology of capacity events
PROMPT_PATH = "prompts/chronology-capacity.txt"
FUNCTION_SCHEMA_PATH = "schemas/chronology-capacity.json"

#process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_chronology_capacity", allowed_types=["capacity"], node_type = "events")


🔹 Processing 1 articles (Offset: 0)
📌 Processing Article ID: 67f52d07981040986eab7b6c
✅ Succesful call to OpenAI. The following nodes are returned: {'nodes': {'companies': [{'id': 'company_volkswagen', 'type': 'company', 'name': 'Volkswagen'}], 'factories': [{'id': 'factory_zwickau', 'type': 'factory', 'name': 'Zwickau', 'location': {'city': 'Zwickau', 'country': 'Germany'}}, {'id': 'factory_wolfsburg', 'type': 'factory', 'name': 'Wolfsburg', 'location': {'city': 'Wolfsburg', 'country': 'Germany'}}, {'id': 'factory_puebla', 'type': 'factory', 'name': 'Puebla', 'location': {'city': 'Puebla', 'country': 'Mexico'}}, {'id': 'factory_emden', 'type': 'factory', 'name': 'Emden', 'location': {'city': 'Emden', 'country': 'Germany'}}], 'products': [{'id': 'product_id3', 'type': 'product', 'name': 'ID.3'}, {'id': 'product_id4', 'type': 'product', 'name': 'ID.4'}, {'id': 'product_id5', 'type': 'product', 'name': 'ID.5'}, {'id': 'product_audi_q4_etron', 'type': 'product', 'name': 'Audi Q4 e-tron'},

In [20]:
articles_collection.find_one({})

{'_id': ObjectId('67f52a45981040986eab706d'),
 'title': 'United Kingdom, Honda + GM, Delaware, Milton Keynes.',
 'paragraphs': [{'p1': '75m for electric transport: The British government and industry awarded almost 75m pounds to five projects. The biggest bulk (46.5m GBP) goes to the London Taxi Corporation to develop electric cabs. Morgan Motors receives funding to develop hybrid- and electric drives for sports cars, while AGM Batteries is working on battery packs. Parker Hannifin looks at electric forklifts and last, Jaguar Land Rover receives 13.1m GBP for turbocharger supply in theUK.',
   'p2': 'gov.uk',
   'p3': 'Joint fuel cell production: Honda and GM will set up a joint fuel cell production facility. The firms have been working together on the technology since 2013 and aim for a mass market eco car. The move is expected to decrease costs. Serial production of fuel cells will start by 2025 at the latest.',
   'p4': 'asahi.com',
   'p5': 'Fuel cell price break: A team from the U

In [42]:
validated_count = articles_collection.count_documents({"validation": True})
print(f"Number of validated articles: {validated_count}")

# If you want to specifically count only electric vehicle articles that are validated:
ev_validated_count = articles_collection.count_documents({
    "validation": True,
    "meta.ID": {"$regex": "^27"}  # Same filter as in your process_relationships function
})
print(f"Number of validated electric vehicle articles: {ev_validated_count}")

Number of validated articles: 46
Number of validated electric vehicle articles: 46


In [29]:
search_for = "relationships_ownership"
search_for = "nodes_ben"

# Count articles with nodes_ben field present
articles_with_nodes = articles_collection.count_documents({search_for: {"$exists": True}})
print(f"Number of articles with nodes_ben field: {articles_with_nodes}")

# If you want to specifically count only electric vehicle articles with nodes_ben:
ev_articles_with_nodes = articles_collection.count_documents({
    search_for: {"$exists": True},
    "meta.ID": {"$regex": "^27"}  # Same filter as in your process_relationships function
})
print(f"Number of electric vehicle articles with {search_for} field: {ev_articles_with_nodes}")


Number of articles with nodes_ben field: 1003
Number of electric vehicle articles with nodes_ben field: 901
